In [1]:
# Install required packages (run once)
!pip install transformers seqeval -q

import os
import sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project root
project_root = '/content/drive/MyDrive/absa-multitask-roberta'
os.chdir(project_root)
sys.path.insert(0, project_root)

# Now imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import json
from utils.helpers import set_seed
from src.model import MultiTaskRoBERTa
from src.train import compute_class_weights, train_epoch, evaluate, evaluate_implicit_explicit

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Current directory:", os.getcwd())
print("Using device:", device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Mounted at /content/drive
Current directory: /content/drive/MyDrive/absa-multitask-roberta
Using device: cuda


In [2]:
with open('data/roberta_data_info.json', 'r') as f:
    info = json.load(f)
max_len = info['max_len']
label2id = info['label2id']
id2label = {int(k): v for k, v in info['id2label'].items()}
num_labels = len(label2id)

train_dict = torch.load('data/roberta_train.pt')
val_dict = torch.load('data/roberta_val.pt')
test_dict = torch.load('data/roberta_test.pt')

In [3]:
class ABSADataset(Dataset):
    def __init__(self, data_dict):
        self.input_ids = data_dict['input_ids']
        self.attention_mask = data_dict['attention_mask']
        self.aspect_labels = data_dict['aspect_labels']
        self.sentiment_labels = data_dict['sentiment_labels']
        self.has_implicit = data_dict['has_implicit']
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.__dict__.items()}

In [4]:
batch_size = 16
train_dataset = ABSADataset(train_dict)
val_dataset = ABSADataset(val_dict)
test_dataset = ABSADataset(test_dict)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

class_weights = compute_class_weights(train_dict['sentiment_labels']).to(device)
aspect_criterion = nn.CrossEntropyLoss(ignore_index=0)
sentiment_criterion = nn.CrossEntropyLoss(weight=class_weights)

In [5]:
model = MultiTaskRoBERTa().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

num_epochs = 10
best_val_f1 = 0.0
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, aspect_criterion, sentiment_criterion, device)
    val_loss, val_f1, val_sent_acc = evaluate(model, val_loader, aspect_criterion, sentiment_criterion, device, id2label)
    print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f} | Aspect F1 {val_f1:.4f} | Sent Acc {val_sent_acc:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'models/multitask_roberta_best.pt')
        print("Saved best model")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Evaluating: 100%|██████████| 22/22 [00:01<00:00, 14.23it/s]


...

Epoch 8: Train Loss 0.0881 | Val Loss 1.5788 | Aspect F1 0.9196 | Sent Acc 0.8029


Evaluating: 100%|██████████| 22/22 [00:01<00:00, 12.45it/s]


Epoch 9: Train Loss 0.0717 | Val Loss 1.5726 | Aspect F1 0.9186 | Sent Acc 0.7971


Evaluating: 100%|██████████| 22/22 [00:01<00:00, 12.37it/s]

Epoch 10: Train Loss 0.0825 | Val Loss 1.5536 | Aspect F1 0.9123 | Sent Acc 0.7943


In [6]:
model.load_state_dict(torch.load('models/multitask_roberta_best.pt'))
test_loss, test_f1, test_sent_acc = evaluate(model, test_loader, aspect_criterion, sentiment_criterion, device, id2label)
print(f"\n===== Test Results =====")
print(f"Aspect F1: {test_f1:.4f}")
print(f"Sentiment Accuracy: {test_sent_acc:.4f}")
imp_f1, exp_f1 = evaluate_implicit_explicit(model, test_loader, device, id2label)
print(f"Implicit F1: {imp_f1:.4f} | Explicit F1: {exp_f1:.4f}")

Evaluating: 100%|██████████| 22/22 [00:01<00:00, 12.32it/s]



===== Test Results =====
Aspect F1: 0.9217
Sentiment Accuracy: 0.7756


Implicit/Explicit: 100%|██████████| 22/22 [00:01<00:00, 12.52it/s]

Implicit F1: 0.9041 | Explicit F1: 0.9318


In [7]:
import numpy as np

seeds = [42, 123, 456, 789, 1010]
results = []   # each element will be (f1, acc)

for seed in seeds:
    set_seed(seed)
    print(f"\n=== Training with seed {seed} ===")

    # Recreate model and optimizer
    model = MultiTaskRoBERTa().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    best_val_f1 = 0.0
    for epoch in range(num_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, aspect_criterion, sentiment_criterion, device)
        val_loss, val_f1, val_sent_acc = evaluate(model, val_loader, aspect_criterion, sentiment_criterion, device, id2label)
        print(f"Epoch {epoch+1}: Val F1 = {val_f1:.4f}, Val Sent Acc = {val_sent_acc:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), f'models/multitask_roberta_seed{seed}.pt')

    # Load best model for this seed and evaluate on test set
    model.load_state_dict(torch.load(f'models/multitask_roberta_seed{seed}.pt'))
    _, test_f1, test_sent_acc = evaluate(model, test_loader, aspect_criterion, sentiment_criterion, device, id2label)
    results.append((test_f1, test_sent_acc))
    print(f"Seed {seed} test F1: {test_f1:.4f}, test Sent Acc: {test_sent_acc:.4f}")

# Compute and print mean±std
f1_scores = [r[0] for r in results]
acc_scores = [r[1] for r in results]
print(f"\n===== Multi‑seed results =====")
print(f"Mean Aspect F1: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"Mean Sentiment Accuracy: {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")


=== Training with seed 42 ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Evaluating: 100%|██████████| 22/22 [00:01<00:00, 12.52it/s]


Epoch 1: Val F1 = 0.9064, Val Sent Acc = 0.7200


Evaluating: 100%|██████████| 22/22 [00:01<00:00, 12.45it/s]


...

Epoch 9: Val F1 = 0.9274, Val Sent Acc = 0.8029


Evaluating: 100%|██████████| 22/22 [00:01<00:00, 12.22it/s]


Epoch 10: Val F1 = 0.9187, Val Sent Acc = 0.7829


Evaluating: 100%|██████████| 22/22 [00:01<00:00, 12.20it/s]

Seed 1010 test F1: 0.9437, test Sent Acc: 0.8097

===== Multi‑seed results =====
Mean Aspect F1: 0.9311 ± 0.0075
Mean Sentiment Accuracy: 0.7705 ± 0.0277


In [ ]:
# Save the final test results to CSV
import pandas as pd

results_df = pd.DataFrame({
    'seed': [42, 123, 456, 789, 1010],
    'test_f1': [0.9217, 0.9280, 0.9346, 0.9273, 0.9437]
})
results_df.to_csv('results/multiseed_results.csv', index=False)
print("Results saved.")

Results saved.
